# Part 5 - Edge Case Testing\n\nStandalone tests that validate the generated raw and cleaned datasets. No imports from Part 1 or Part 2 notebooks are required.

In [1]:

from pathlib import Path
import unittest
import pandas as pd

# --------------------------------------------------
# Paths (Notebook should be in Part 5/testing)
# --------------------------------------------------
BASE = Path.cwd()
RAW_DIR = (BASE / "../../Part 1/raw").resolve()
CLEAN_DIR = (BASE / "../../Part 2/cleaned_data").resolve()

orders = pd.read_csv(RAW_DIR / "orders.csv")
order_items = pd.read_csv(RAW_DIR / "order_items.csv")
clean_orders = pd.read_csv(CLEAN_DIR / "orders_clean.csv")

class TestEdgeCases(unittest.TestCase):

    def test_orphan_order_ids(self):
        valid_ids = set(orders["order_id"])
        orphans = order_items[~order_items["order_id"].isin(valid_ids)]
        self.assertGreater(len(orphans), 0,
                           "No orphan order_id rows found.")

    def test_discount_over_100(self):
        bad = order_items[order_items["discount_percent"] > 100]
        self.assertGreater(len(bad), 0,
                           "No discount >100 rows found.")
        revenue = bad["quantity"] * bad["unit_price"] * (1 - bad["discount_percent"]/100)
        self.assertTrue((revenue < 0).all())

    def test_zero_quantity(self):
        zero = order_items[order_items["quantity"] == 0]
        self.assertGreater(len(zero), 0,
                           "No zero-quantity rows found.")
        revenue = zero["quantity"] * zero["unit_price"] * (1 - zero["discount_percent"]/100)
        self.assertTrue((revenue == 0).all())

    def test_future_dated_orders_flagged(self):
        self.assertIn("is_future_dated", clean_orders.columns,
                      "orders_clean.csv is missing 'is_future_dated' column.")
        future = clean_orders[clean_orders["is_future_dated"] == True]
        self.assertGreater(len(future), 0,
                           "No future-dated orders were flagged.")

suite = unittest.defaultTestLoader.loadTestsFromTestCase(TestEdgeCases)
runner = unittest.TextTestRunner(verbosity=2)
runner.run(suite)


test_discount_over_100 (__main__.TestEdgeCases.test_discount_over_100) ... ok
test_future_dated_orders_flagged (__main__.TestEdgeCases.test_future_dated_orders_flagged) ... ok
test_orphan_order_ids (__main__.TestEdgeCases.test_orphan_order_ids) ... ok
test_zero_quantity (__main__.TestEdgeCases.test_zero_quantity) ... ok

----------------------------------------------------------------------
Ran 4 tests in 0.035s

OK


<unittest.runner.TextTestResult run=4 errors=0 failures=0>